# Inspecting final stream results and table relationships

Query retained native SQLite results and the original small Spark Parquet files. This inspection does not rerun Flink or rebuild the warehouse. The Flink topology screenshots show pair B while this notebook explicitly reads the later complete run D, `mentor-stream-20261101d`.

Requires Python, pandas, DuckDB 1.4.3 and IPython. Run from a checkout with `data/lakehouse/day2-final` restored for the table sections. Stream snapshots come directly from the bundled archive. Connections are read-only; all source bytes are checked before and after inspection.

In [ ]:
from pathlib import Path
from hashlib import sha256
import json, sqlite3, tarfile, tempfile
from html import escape
import pandas as pd
import duckdb
from IPython.display import display, SVG

ROOT = Path.cwd()
while not (ROOT / 'pipelines/transforms.py').is_file():
    if ROOT == ROOT.parent:
        raise FileNotFoundError('Open inside the project checkout')
    ROOT = ROOT.parent
EVIDENCE = ROOT / 'docs/evidence/data-platform-review'
bundle = EVIDENCE / 'stream-entrypoint/native.tar.gz'
report_path = EVIDENCE / 'stream-entrypoint/verification.json'
report = json.loads(report_path.read_bytes())
original_pins = {str(p.relative_to(ROOT)): sha256(p.read_bytes()).hexdigest() for p in (bundle, report_path)}
temporary = tempfile.TemporaryDirectory(prefix='stream-readback-', dir=ROOT / 'outputs')
stores, store_pins = {}, {}
with tarfile.open(bundle) as archive:
    for variant in ['baseline', 'corrected', 'retention-probe']:
        member = archive.getmember(f'data/stream/mentor-stream-20261101d/{variant}.sqlite')
        assert member.isfile() and member.size < 1_000_000
        body = archive.extractfile(member).read()
        path = Path(temporary.name) / (variant + '.sqlite')
        path.write_bytes(body)
        store_pins[variant] = sha256(body).hexdigest()
        stores[variant] = sqlite3.connect(path.as_uri() + '?mode=ro&immutable=1', uri=True)
        assert stores[variant].execute('PRAGMA integrity_check').fetchone()[0] == 'ok'
display(pd.DataFrame([{'variant': v, 'job_id': report[v.replace('-', '_')]['runtime']['job_id'],
    'records': report[v.replace('-', '_')]['runtime']['records'],
    'watermark_delay_min': report[v.replace('-', '_')]['runtime']['watermark_delay_minutes']}
    for v in stores]))
print('Retained run D | SQLite snapshots authenticated from bundled native archive')

## Window values before and after correction

The baseline has no deduplication and zero watermark delay. The corrected job changes both: a 15-minute delay plus an event cache/ledger. The joint correction recovers D and removes repeated readings. These are persisted final results, not live Flink counters or a performance comparison.

In [ ]:
window_rows = []
for variant in ['baseline', 'corrected']:
    database = stores[variant]
    rows = [json.loads(row[0]) for row in database.execute('SELECT payload FROM windows ORDER BY machine_id,window_start')]
    assert rows == report[variant]['windows']
    for row in rows:
        window_rows.append({'variant': variant, 'machine': row['machine_id'],
            'window_start_utc': row['window_start'][11:16], 'window_end_utc': row['event_timestamp'][11:16],
            'readings': row['reading_count'], 'temperature_sum': row['temperature_sum'],
            'temperature_mean': row['temperature_mean']})
windows = pd.DataFrame(window_rows)
assert windows.readings.tolist() == [6, 2, 2, 3, 1, 1]
display(windows)
print('A+B+D: count 3, sum 60 | C at 00:10 starts the next half-open window | P-002 stays isolated')

## Late arrival: D is recovered, E remains quarantined

Inspect the keyed late-event audit and actual observed watermark barriers. Baseline late delivery occurrences and corrected late event IDs have different grains. Do not interpret them as a 6-to-1 improvement rate.

In [ ]:
late_rows, barriers = [], []
for variant in ['baseline', 'corrected']:
    db = stores[variant]
    late_ids = [r[0] for r in db.execute("SELECT json_extract(payload,'$.event.event_id') FROM audits WHERE kind='late' ORDER BY identity")]
    assert late_ids == report[variant]['late_event_ids']
    late_rows.append({'variant': variant, 'distinct_late_event_ids': ', '.join(late_ids),
        'D_quarantined': 'D' in late_ids, 'E_quarantined': 'E' in late_ids})
    for payload, in db.execute("SELECT payload FROM audits WHERE kind='progress' ORDER BY identity"):
        row = json.loads(payload)
        barriers.append({'variant': variant, 'barrier': row['marker'],
            'event_time_utc': row['event_ts'][11:19],
            'observed_watermark_utc': pd.to_datetime(row['watermark_ms'], unit='ms', utc=True).strftime('%Y-%m-%d %H:%M:%S.%f')[:-3]})
display(pd.DataFrame(late_rows))
display(pd.DataFrame(barriers))
print('D: event minute 8, first arrival minute 16. E: event minute 7, first arrival minute 32.')

## Retry/replay and cache expiry

The scenario-lifetime disk ledger protects IDs after the finite cache expires. It is not a checkpoint-transaction sink or a general crash-recovery guarantee. The 1.5-second retention probe is a separate job, not a timing comparison against the baseline.

In [ ]:
audit_rows = []
for variant, db in stores.items():
    counts = dict(db.execute('SELECT kind,count(*) FROM audits GROUP BY kind'))
    result = report[variant.replace('-', '_')]
    # Baseline's late report counts delivery occurrences; audits have stable per-event identities.
    for key in ['unique', 'duplicate', 'expired']:
        assert counts.get(key, 0) == result['audit_counts'].get(key, 0)
    assert [json.loads(r[0]) for r in db.execute('SELECT payload FROM windows ORDER BY machine_id,window_start')] == result['windows']
    audit_rows.append({'variant': variant, 'unique_ID_audits': counts.get('unique', 0),
        'duplicate_delivery_audits': counts.get('duplicate', 0), 'cache_expirations': counts.get('expired', 0),
        'ledger_duplicates': result['duplicates_from_durable_ledger'],
        'final_total_readings': sum(w['reading_count'] for w in result['windows'])})
display(pd.DataFrame(audit_rows))
assert audit_rows[1]['duplicate_delivery_audits'] == 13
assert audit_rows[2]['cache_expirations'] == 2 and audit_rows[2]['final_total_readings'] == 1
print('Corrected: 13 duplicate deliveries rejected. Retention probe: final count 1 after expiry and replay.')

## Persisted tables in every zone

This is the original `day2-final` layout, not the later DP1-DP3 pilot. Bronze stores one envelope table, Silver has four typed tables, and Gold stores dimensions, facts, features, labels and the eligible training subset. The inventory and types below come from the actual Parquet files.

In [ ]:
source = ROOT / 'data/lakehouse/day2-final'
pin_path = EVIDENCE / 'schema/input-files.json'
pins = json.loads(pin_path.read_bytes())['files']
original_pins[str(pin_path.relative_to(ROOT))] = sha256(pin_path.read_bytes()).hexdigest()
for pin in pins:
    p = source / pin['path']
    assert p.stat().st_size == pin['bytes'] and sha256(p.read_bytes()).hexdigest() == pin['sha256']
    original_pins[str(p.relative_to(ROOT))] = pin['sha256']
con = duckdb.connect(':memory:', config={'threads': 2, 'memory_limit': '1GB'})
inventory, schemas = [], {}
for zone in ['bronze', 'silver', 'gold']:
    con.execute(f'CREATE SCHEMA {zone}')
    for folder in sorted((source / zone).iterdir()):
        parts = sorted(folder.glob('*.parquet')) if folder.is_dir() else []
        if not parts: continue
        paths = '[' + ','.join("'" + p.as_posix().replace("'", "''") + "'" for p in parts) + ']'
        name = zone + '.' + folder.name
        con.execute(f'CREATE VIEW {name} AS SELECT * FROM read_parquet({paths})')
        rows = con.execute(f'SELECT count(*) FROM {name}').fetchone()[0]
        columns = con.execute(f'DESCRIBE {name}').fetchall()
        schemas[name] = {r[0]: r[1] for r in columns}
        inventory.append({'zone': zone, 'table': folder.name, 'rows': rows, 'columns': len(columns)})
display(pd.DataFrame(inventory))
print(f'DuckDB {duckdb.__version__} | {len(pins)} unchanged original Parquet files')

## Query-derived fact/dimension relationship diagram

Each arrow points from a child table to its parent key. Counts, key uniqueness, null keys and missing parents are queried before drawing. This is a visualization of verified logical joins, not SQL-enforced foreign keys; Parquet has no such constraints. The fact's `machine_sk` was assigned by the pipeline's half-open event-time SCD2 join.

In [ ]:
relationships = []
for child, parent, key in [('fact_telemetry','dim_machine','machine_sk'),
    ('fact_failure','dim_machine','machine_sk'), ('fact_maintenance','dim_machine','machine_sk'),
    ('dim_machine','dim_factory','factory_id')]:
    assert key in schemas['gold.' + child] and key in schemas['gold.' + parent]
    count, missing, nulls = con.execute(f"SELECT count(*), count(*) FILTER (WHERE p.{key} IS NULL), count(*) FILTER (WHERE c.{key} IS NULL) FROM gold.{child} c LEFT JOIN gold.{parent} p USING({key})").fetchone()
    child_count = con.execute(f'SELECT count(*) FROM gold.{child}').fetchone()[0]
    unique = con.execute(f'SELECT count(*) = count(DISTINCT {key}) FROM gold.{parent}').fetchone()[0]
    assert unique and count == child_count and missing == nulls == 0
    relationships.append({'child': child, 'parent': parent, 'key': key, 'child_rows': count, 'missing_parent': missing, 'null_key': nulls})
counts = {r['table']: r['rows'] for r in inventory}
positions = {'fact_telemetry': (15,20), 'fact_failure': (15,132), 'fact_maintenance': (15,244),
    'dim_machine': (385,132), 'dim_factory': (755,132)}
svg = ['<svg xmlns="http://www.w3.org/2000/svg" width="1030" height="350" viewBox="0 0 1030 350" role="img" aria-label="Query-derived Gold relationships">',
    '<defs><marker id="join-end" markerWidth="8" markerHeight="8" refX="7" refY="4" orient="auto"><path d="M0,0 L8,4 L0,8 Z" fill="#2563eb"/></marker></defs>',
    '<rect width="1030" height="350" fill="#fff"/>']
for row in relationships:
    x,y = positions[row['child']]; px,py = positions[row['parent']]
    mid = (x + 250 + px) / 2
    svg.append(f'<path d="M{x+250},{y+40} H{mid} V{py+40} H{px-8}" stroke="#2563eb" stroke-width="2" fill="none" marker-end="url(#join-end)"/>')
for table,(x,y) in positions.items():
    key = 'factory_id' if table == 'dim_factory' else 'machine_sk'
    color = '#ecfdf5' if table.startswith('dim_') else '#eff6ff'
    svg.append(f'<rect x="{x}" y="{y}" width="250" height="80" rx="9" fill="{color}" stroke="#94a3b8"/>')
    for offset,text in [(24,table),(46,f'{counts[table]:,} rows'),(66,key + (' [unique]' if table.startswith('dim_') else ' [logical FK]'))]:
        svg.append(f'<text x="{x+12}" y="{y+offset}" font-family="sans-serif" font-size="15" fill="#0f172a">{escape(text)}</text>')
svg.append('<text x="20" y="342" font-family="sans-serif" font-size="13" fill="#475569">Child → parent | every parent key unique | no null or missing keys | source: day2-final Parquet</text></svg>')
relationship_svg = ''.join(svg)
display(SVG(relationship_svg))
display(pd.DataFrame(relationships))

## From normalized records to snapshots and training

Silver-to-fact `event_id` equality and machine-version membership are separate relationships. Feature and label tables each have a unique `(machine_id,event_timestamp)` grain; the training table is a filtered, purged subset. These checks do not interpret future labels as feature inputs.

In [ ]:
links = []
for silver, fact, key in [('stg_telemetry','fact_telemetry','event_id'), ('stg_failure','fact_failure','failure_id'), ('stg_maintenance','fact_maintenance','maintenance_id')]:
    rows, unique = con.execute(f'SELECT count(*),count(DISTINCT {key}) FROM silver.{silver}').fetchone()
    delta = con.execute(f'(SELECT {key} FROM silver.{silver} EXCEPT ALL SELECT {key} FROM gold.{fact}) UNION ALL (SELECT {key} FROM gold.{fact} EXCEPT ALL SELECT {key} FROM silver.{silver})').fetchall()
    assert not delta and rows == unique
    links.append({'source': 'silver.' + silver, 'destination': 'gold.' + fact, 'key': key, 'rows': rows, 'unmatched_keys': len(delta)})
for table in ['feat_machine_10m', 'label_failure_1h', 'training_failure_1h']:
    rows, unique = con.execute(f'SELECT count(*), count(DISTINCT (machine_id,event_timestamp)) FROM gold.{table}').fetchone()
    assert rows == unique
    for target in (['label_failure_1h'] if table == 'feat_machine_10m' else ['feat_machine_10m']):
        unmatched = con.execute(f'SELECT count(*) FROM gold.{table} c LEFT JOIN gold.{target} p USING(machine_id,event_timestamp) WHERE p.machine_id IS NULL').fetchone()[0]
        assert unmatched == 0
        links.append({'source': 'gold.' + table, 'destination': 'gold.' + target, 'key': 'machine_id,event_timestamp', 'rows': rows, 'unmatched_keys': unmatched})
display(pd.DataFrame(links))
for p,h in original_pins.items():
    assert sha256((ROOT / p).read_bytes()).hexdigest() == h, p
for variant, db in stores.items():
    db.close()
    assert sha256((Path(temporary.name)/(variant+'.sqlite')).read_bytes()).hexdigest() == store_pins[variant]
con.close()
temporary.cleanup()
print('PASS: all original archive/report/Parquet bytes unchanged; no warehouse or Flink job was rerun')